In [1]:
import requests
import csv
import datetime
import time
import pandas as pd
import joblib
import numpy as np
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from transformers import T5ForConditionalGeneration
from transformers import T5Tokenizer, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

In [2]:
def get_PBP(year, year2, url):
  team_data_path = f'/content/drive/MyDrive/CS465_Project/CSV_Data_Files/NHL/MY_Data/{year}_{year2}_All_Teams_Data_TEST (1).csv'

  inputText = ""
  outputText = ""

  res = requests.get(url)
  data = res.json()

  away = data.get('awayTeam', [])
  home = data.get('homeTeam', [])

  inputText += f' {home["abbrev"]}'

  home_team_df = pd.read_csv(team_data_path)
  ht_index = home_team_df.index[home_team_df["team"] == home["abbrev"]]
  if len(ht_index) > 0:
    ht_index = ht_index[0]
    inputText += f'\n\t{home_team_df.loc[ht_index, "gamesPlayed"]} {home_team_df.loc[ht_index, "wins"]} {home_team_df.loc[ht_index, "losses"]} '
    inputText += f'\n\t{home_team_df.loc[ht_index, "xGoalsFor"]} {home_team_df.loc[ht_index, "xReboundsFor"]} {home_team_df.loc[ht_index, "xShotAttemptsPercentage"]} '
    inputText += f'\n\t{home_team_df.loc[ht_index, "shotsOnGoalFor"]} {home_team_df.loc[ht_index, "blockedShotAttemptsFor"]} {home_team_df.loc[ht_index, "penaltiesFor"]} '
    inputText += f'\n\t{home_team_df.loc[ht_index, "faceOffsWonFor"] / (home_team_df.loc[ht_index, "faceOffsLostFor"] + home_team_df.loc[ht_index, "faceOffsWonFor"])} '

    inputText += f'{home_team_df.loc[ht_index,"hitsFor"]} {home_team_df.loc[ht_index,"takeaways"]} {home_team_df.loc[ht_index,"giveaways"]} '

    inputText += f'\n\t{home_team_df.loc[ht_index,"gamesPlayed"]} {home_team_df.loc[ht_index,"wins"]} {home_team_df.loc[ht_index,"losses"]} '
    inputText += f'\n\t{home_team_df.loc[ht_index,"xGoalsFor"]} {home_team_df.loc[ht_index,"xReboundsFor"]} {home_team_df.loc[ht_index,"xShotAttemptsPercentage"]} '
    inputText += f'\n\t{home_team_df.loc[ht_index,"shotsOnGoalFor"]} {home_team_df.loc[ht_index,"blockedShotAttemptsFor"]} {home_team_df.loc[ht_index,"penaltiesFor"]} '

    inputText += f' {away["abbrev"]}'

  away_team_df = pd.read_csv(team_data_path)
  at_index = away_team_df.index[away_team_df["team"] == away["abbrev"]]
  if len(at_index) > 0:
    at_index = at_index[0]
    inputText += f'\n\t{away_team_df.loc[at_index, "gamesPlayed"]} {away_team_df.loc[at_index, "wins"]} {away_team_df.loc[at_index, "losses"]} '
    inputText += f'\n\t{away_team_df.loc[at_index, "xGoalsFor"]} {away_team_df.loc[at_index, "xReboundsFor"]} {away_team_df.loc[at_index, "xShotAttemptsPercentage"]} '
    inputText += f'\n\t{away_team_df.loc[at_index, "shotsOnGoalFor"]} {away_team_df.loc[at_index, "blockedShotAttemptsFor"]} {away_team_df.loc[at_index, "penaltiesFor"]} '
    inputText += f'\n\t{away_team_df.loc[at_index, "faceOffsWonFor"] / (away_team_df.loc[at_index, "faceOffsLostFor"] + away_team_df.loc[at_index, "faceOffsWonFor"])} '

    inputText += f'{away_team_df.loc[at_index, "hitsFor"]} {away_team_df.loc[at_index, "takeaways"]} {away_team_df.loc[at_index, "giveaways"]} '

    inputText += f'\n\t{away_team_df.loc[at_index, "gamesPlayed"]} {away_team_df.loc[at_index, "wins"]} {away_team_df.loc[at_index, "losses"]} '
    inputText += f'\n\t{away_team_df.loc[at_index, "xGoalsFor"]} {away_team_df.loc[at_index, "xReboundsFor"]} {away_team_df.loc[at_index, "xShotAttemptsPercentage"]} '
    inputText += f'\n\t{away_team_df.loc[at_index, "shotsOnGoalFor"]} {away_team_df.loc[at_index, "blockedShotAttemptsFor"]} {away_team_df.loc[at_index, "penaltiesFor"]} '

  teams_dict = {
        1 : "NJD", 2 : "NYI", 3 : "NYR",
        4 : "PHI", 5 : "PIT", 6 : "BOS",
        7 : "BUF", 8 : "MTL", 9 : "OTT",
        10 : "TOR", 12 : "CAR", 13 : "FLA",
        14 : "TBL", 15 : "WSH", 16 : "CHI",
        17 : "DET", 18 : "NSH", 19 : "STL",
        20 : "CGY", 21 : "COL", 22 : "EDM",
        23 : "VAN", 24 : "ANA", 25 : "DAL",
        26 : "LAK", 27 : "PHX", 28 : "SJS",
        29 : "CBJ", 30 : "MIN", 52 : "WPG",
        53 : "ARI", 54 : "VGK", 55 : "SEA",
        59 : "UTA"

    }
  count = 0
  randSplit = np.random.randint(0, len(data.get("plays", [])))

  for event in data.get("plays", []):
    if count <= randSplit:
      count += 1
      match event["typeCode"]:
        case 502:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : FaceOff won by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 503:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Hit by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 504:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Giveaway by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 505:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Goal by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 506:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Shot on Goal by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 507:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Missed Shot by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 508:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Blocked Shot by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 509:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Penalty by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 525:
          inputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Takeaway by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
    else:
      count += 1
      match event["typeCode"]:
        case 502:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : FaceOff won by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 503:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Hit by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 504:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Giveaway by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 505:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Goal by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 506:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Shot on Goal by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 507:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Missed Shot by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 508:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Blocked Shot by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 509:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Penalty by {teams_dict[event["details"]["eventOwnerTeamId"]]}'
        case 525:
          outputText += f'\n {event["timeInPeriod"]} in Period {event["periodDescriptor"]["number"]} : Takeaway by {teams_dict[event["details"]["eventOwnerTeamId"]]}'

  outputText += f'\nScore:\n\t{home["abbrev"]}: {home["score"]}\n\t{away["abbrev"]}: {away["score"]}'
  outputText += f'\nShots On Goal:\n\t{home["abbrev"]}: {home["sog"]}\n\t{away["abbrev"]}: {away["sog"]}'

  return inputText, outputText


def train_model(all_inputs, all_outputs):
    tokenizer = T5Tokenizer.from_pretrained("t5-small")

    # Define custom dataset
    class HockeyDataset(Dataset):
        def __init__(self, inputs, targets, tokenizer):
            self.inputs = inputs
            self.targets = targets
            self.tokenizer = tokenizer

        def __len__(self):
            return len(self.inputs)

        def __getitem__(self, idx):
            input_text = self.inputs[idx]
            target_text = self.targets[idx]

            input_encodings = self.tokenizer(
                input_text, padding="max_length", truncation=True, max_length=512, return_tensors="pt"
            )
            target_encodings = self.tokenizer(
                target_text, padding="max_length", truncation=True, max_length=128, return_tensors="pt"
            )

            return {
                "input_ids": input_encodings["input_ids"].squeeze(),
                "attention_mask": input_encodings["attention_mask"].squeeze(),
                "labels": target_encodings["input_ids"].squeeze()
            }

    # Load dataset
    train_dataset = HockeyDataset(all_inputs, all_outputs, tokenizer)

    # Load model
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Training settings
    training_args = TrainingArguments(
        output_dir="./t5_hockey_model",
        per_device_train_batch_size=4,
        num_train_epochs=3,
        logging_dir="./logs",
        save_total_limit=2,
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset
    )

    print("Training started...")
    trainer.train()
    print("Training completed!")

    # Save trained model
    model.save_pretrained("./t5_hockey_model")
    tokenizer.save_pretrained("./t5_hockey_model")

def main():
  gameid = 1
  year = 2022

  all_Inputs = []
  all_Outputs = []

  while gameid < 1000:
    url = f'https://api-web.nhle.com/v1/gamecenter/{year}02{gameid:04d}/play-by-play'
    inputText, outputText = get_PBP(year, year+1, url)
    gameid += 1
    all_Inputs.append(inputText)
    all_Outputs.append(outputText)

  train_model(all_Inputs, all_Outputs)

main()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CS465_Project/CSV_Data_Files/NHL/MY_Data/2022_2023_All_Teams_Data_TEST (1).csv'